# 第 12 章 · LQR：模型已知、线性、二次代价时，Bellman 方程有闭式解

配套讲义：仓库里 `robot-motion-rl/notes/lessons/12-lqr.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（系统和权重都是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 线性化 | g/l | `linearize_pendulum()` | 9.8 |
| 第三步 Riccati | 反复代入 | `solve_lqr(1.1, 1, 1, 1)` | P：1、1.605、1.746、1.769 → 1.774；K = 0.703 |
| 第三步 闭环 | (a − bK)ˣ | `simulate(1.1, 1, K, 1, 5)` | 5 步后 0.0098 |
| 第四步 调权重 | 同上 | `solve_lqr(1.1, 1, q, r)` | K = 0.370、1.009 |

例子（编的）：一维离散系统 x_{t+1} = a·x_t + b·u_t，代价 Σ (q·x² + r·u²)。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `linearize_pendulum`**：倒立摆在竖直附近：θ̈ = (g/l)·sin θ + u ≈ (g/l)·θ + u，返回线性系数 g/l。

In [ ]:
def linearize_pendulum(g=9.8, length=1.0):
    """倒立摆在竖直附近：θ̈ = (g/l)·sin θ + u ≈ (g/l)·θ + u，返回线性系数 g/l。"""
    return g / length

**函数 `riccati_step`**：一步 Riccati 迭代：P ← q + a²P − (abP)² / (r + b²P)。

In [ ]:
def riccati_step(p, a, b, q, r):
    """一步 Riccati 迭代：P ← q + a²P − (abP)² / (r + b²P)。"""
    return q + a * a * p - (a * b * p) ** 2 / (r + b * b * p)

**函数 `solve_lqr`**：反复迭代到收敛，返回 (P, K, 每次迭代的 P)。K = abP / (r + b²P)，控制律 u = −K·x。

In [ ]:
def solve_lqr(a, b, q, r, iters=200):
    """反复迭代到收敛，返回 (P, K, 每次迭代的 P)。K = abP / (r + b²P)，控制律 u = −K·x。"""
    p, hist = q, [q]
    for _ in range(iters):
        p = riccati_step(p, a, b, q, r)
        hist.append(p)
    k = a * b * p / (r + b * b * p)
    return p, k, hist

**函数 `simulate`**：闭环 x_{t+1} = (a − bK)·x_t，返回轨迹。

In [ ]:
def simulate(a, b, k, x0, steps):
    """闭环 x_{t+1} = (a − bK)·x_t，返回轨迹。"""
    xs = [x0]
    for _ in range(steps):
        xs.append((a - b * k) * xs[-1])
    return np.array(xs)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 线性化")
print("θ̈ ≈ (g/l)·θ + u，g/l =", linearize_pendulum())

In [ ]:
print("\n第三步 Riccati 迭代（a=1.1, b=1, q=1, r=1）")
p, k, hist = solve_lqr(1.1, 1.0, 1.0, 1.0)
print("前 4 次 P：", [round(v, 4) for v in hist[:4]])
print(f"收敛 P = {p:.4f}，K = {k:.4f}，闭环 a − bK = {1.1 - k:.4f}")

In [ ]:
print("\n第四步 调 q 和 r")
for q, r in [(1, 1), (1, 10), (10, 1)]:
    _, k, _ = solve_lqr(1.1, 1.0, q, r)
    xs = simulate(1.1, 1.0, k, 1.0, 5)
    print(f"q={q}, r={r}: K = {k:.3f}，闭环系数 {1.1 - k:.3f}，5 步后 x = {xs[-1]:.4f}")

## 3. 代码里才看得到的

- 迭代从 P = q 开始，相当于「只剩最后一步」时的价值；迭代 200 次是为了保险，实际 10 次以内就和收敛值差不到 0.001。

## 4. 自己改一改

- 把 b 从 1 改成 0.1（控制效果弱），看 K 和闭环系数怎么变。
- 把 a 改成 2.0，看 r 要小到多少闭环才能稳。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 院子当平地，平衡点附近拉直线；
> 二次代价配线性模型，价值是抛物线、策略是一条直线；
> Riccati 反复代，几步就收敛，P 是价值、K 是策略；
> q 在乎偏离、r 舍不得用力，离平衡点远了就不灵。